# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset described using the [Croissant schema](https://mlcommons.org/croissant/), specifically using [mlcroissant](https://github.com/mlcommons/croissant) in Python.

## Dataset Source
The dataset is published at:
- [`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

> The dataset contains protein abundance and modification data from mass spectrometry of extracellular vesicles from human mast cells.

In [ ]:
# Ensure mlcroissant is installed in the environment
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and preview its RDF structure using the mlcroissant library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
List record sets and their fields/columns using their `@id`s, as described by the Croissant metadata.

*Note: All references use entity `@id` values; this ensures all field/record set references are robust and persist across schema updates.*

In [ ]:
def summarize_record_sets(metadata):
    print("Available Record Sets in the Dataset:")
    if hasattr(metadata, 'record_sets') and metadata.record_sets:
        for recset in metadata.record_sets:
            print(f"- RecordSet @id: {recset['@id']}")
            fields = recset.get('fields', [])
            if fields:
                print("  Fields (by @id):")
                for fld in fields:
                    print(f"    - {fld['@id']}")
            columns = recset.get('columns', [])
            if columns:
                print("  Columns (by @id):")
                for col in columns:
                    print(f"    - {col['@id']}")
    else:
        # If record_sets is not a parsed property, attempt from .to_json())
        json_md = metadata.to_json() if hasattr(metadata, 'to_json') else metadata
        if isinstance(json_md, dict) and 'recordSet' in json_md and json_md['recordSet']:
            for rset in json_md['recordSet']:
                # Each rset could be an @id reference or dict
                if isinstance(rset, dict) and '@id' in rset:
                    print(f"- RecordSet @id: {rset['@id']}")
                elif isinstance(rset, str):
                    print(f"- RecordSet @id: {rset}")
        else:
            print("No record sets found in metadata.")

summarize_record_sets(metadata)

### Preview records from each Record Set
Replace `<record_set_id>` with the specific `@id` of the record set you wish to enumerate.

Below, we show the usage for the likely main data RecordSet (as found in the Croissant schema, if present):

In [ ]:
# Find the main record set id(s). We'll auto-detect if not referenced in metadata.record_sets.
import json
md_json = metadata.to_json() if hasattr(metadata, 'to_json') else metadata

# Helper to retrieve record set ids
record_set_ids = []
if 'recordSet' in md_json:
    if isinstance(md_json['recordSet'], list):
        for rset in md_json['recordSet']:
            if isinstance(rset, dict) and '@id' in rset:
                record_set_ids.append(rset['@id'])
            elif isinstance(rset, str):
                record_set_ids.append(rset)
    elif isinstance(md_json['recordSet'], dict) and '@id' in md_json['recordSet']:
        record_set_ids.append(md_json['recordSet']['@id'])
    elif isinstance(md_json['recordSet'], str):
        record_set_ids.append(md_json['recordSet'])

print("Detected record set @ids:", record_set_ids)

# Preview a handful of records for each record set
if record_set_ids:
    for recset_id in record_set_ids:
        print(f"Sample records from RecordSet {recset_id}:")
        try:
            for i, rec in enumerate(dataset.records(record_set=recset_id)):
                print(json.dumps(rec, indent=2))
                if i >= 2:  # Show only 3 samples
                    break
        except Exception as e:
            print(f"Error reading records for {recset_id}: {e}")
else:
    print("No record sets with records found in schema. Check metadata for data files or sources.")

## 3. Data Extraction
Load records from each valid RecordSet (using its `@id`) into a Pandas DataFrame for analysis.

In [ ]:
# Build DataFrames for each record set - all entities referenced BY '@id'
dfs = {}
for recset_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=recset_id))
        if recs:
            dfs[recset_id] = pd.DataFrame(recs)
            print(f"Loaded {len(dfs[recset_id])} records from RecordSet {recset_id}")
        else:
            print(f"RecordSet {recset_id} yielded no records.")
    except Exception as e:
        print(f"Could not load records for RecordSet {recset_id}: {e}")

# List columns for the primary record set
if dfs:
    main_id = list(dfs.keys())[0]
    print(f"Columns in primary RecordSet ({main_id}):\n{dfs[main_id].columns.tolist()}")
    dfs[main_id].head()
else:
    print("No tabular data loaded from detected record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA operations: filter records, normalize numeric fields, and group by categorical fields.

All field and record set references use their Croissant `@id`.

We'll choose an appropriate numeric field and group field from the DataFrame columns:

In [ ]:
# Pick field names (by Croissant @id or mapped column name) for numeric and group analysis
import numpy as np

if dfs:
    main_id = list(dfs)[0]
    df = dfs[main_id].copy()

    # Attempt to identify numeric fields (float or int columns)
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_candidates) == 0:
        # Try to parse numerics from object columns
        for col in df.columns:
            try:
                df[col + "_numeric"] = pd.to_numeric(df[col], errors='coerce')
                if df[col + "_numeric"].notnull().sum() > 0:
                    numeric_candidates.append(col + "_numeric")
            except Exception:
                continue
    print(f"Detected numeric candidates: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using '{numeric_field}' as numeric_field for EDA.")
    else:
        print("No suitable numeric field found. EDA steps are skipped.")
        numeric_field = None

    # Attempt to pick a group field (categorical/text column)
    group_candidates = [c for c in df.columns if df[c].dtype == "object" and c != numeric_field]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        print(f"Using '{group_field}' as group_field.")

    # Proceed with filtering, normalization, grouping EDA
    if numeric_field:
        try:
            threshold = np.nanmedian(df[numeric_field])
        except Exception:
            threshold = 1
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered to {len(filtered_df)} records where {numeric_field} > {threshold}")

        # Normalization
        if filtered_df[numeric_field].std() > 0:
            filtered_df[numeric_field + "_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / filtered_df[numeric_field].std()
        else:
            filtered_df[numeric_field + "_normalized"] = 0
        print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Grouping
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA analysis.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and summarize by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and 'numeric_field' in locals() and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='teal')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Barplot grouped mean if group_field exists
    if 'group_field' in locals() and group_field and group_field in df:
        grp = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:10] # top 10
        plt.figure(figsize=(8, 5))
        sns.barplot(x=grp.values, y=grp.index, palette='mako')
        plt.title(f'Mean {numeric_field} by {group_field} (top 10)')
        plt.xlabel(f'Mean {numeric_field}')
        plt.ylabel(group_field)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion

We've loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the mlcroissant library, referenced all data elements by their Croissant `@id`, and demonstrated how to extract, preprocess, and visualize data.

#### Key learnings:
- Croissant `@id` provides a stable method to reference any entity within the dataset schema and is used in all loading/querying.
- The dataset contains tabular records for protein analysis that can be used for downstream statistical or ML applications.
- Using `mlcroissant`, you can reproducibly load standardized datasets and programmatically explore their schema and content.
